In [0]:
public_transport_path = "/Volumes/workspace/default/urbanpulse_bronze/public_transport"

pt_files = dbutils.fs.ls(public_transport_path)

for file in pt_files:
    print(file.name, "|", file.size, "bytes")

In [0]:
from pyspark.sql import functions as F

pt_data = {}

for filename in ["stops.txt", "routes.txt", "trips.txt", "stop_times.txt"]:

    name = filename.replace(".txt", "")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{public_transport_path}/{filename}")
    )

    pt_data[name] = df

    print(f"\n{'=' * 50}")
    print("DATASET:", name)
    print("Rows:", df.count())
    print("Columns:", df.columns)
    df.printSchema()
    display(df.limit(5))

In [0]:
from pyspark.sql import functions as F

for name, df in pt_data.items():
    print(f"\n--- {name.upper()} ---")

    print("Total records:", df.count())
    print("Duplicate records:", df.count() - df.dropDuplicates().count())

    null_summary = df.select([
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ])

    display(null_summary)

In [0]:
from pyspark.sql import functions as F

for name, df in pt_data.items():
    total_rows = df.count()

    missing_cells = sum(
        df.filter(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == "")
        ).count()
        for c in df.columns
    )

    duplicate_rows = total_rows - df.dropDuplicates().count()

    print(
        f"{name}: Rows={total_rows}, "
        f"Duplicates={duplicate_rows}, "
        f"Missing cells={missing_cells}"
    )

In [0]:
routes_df = pt_data["routes"]

for column in routes_df.columns:
    missing = routes_df.filter(
        F.col(column).isNull() |
        (F.trim(F.col(column).cast("string")) == "")
    ).count()

    if missing > 0:
        print(f"{column}: {missing} missing values")

display(routes_df.limit(10))

In [0]:
from pyspark.sql import functions as F

clean_pt_data = {}

for name, df in pt_data.items():

    # Remove columns that are completely empty
    empty_columns = []

    for column in df.columns:
        if df.filter(
            F.col(column).isNotNull() &
            (F.trim(F.col(column).cast("string")) != "")
        ).limit(1).count() == 0:
            empty_columns.append(column)

    if empty_columns:
        df = df.drop(*empty_columns)

    # Standardize column names
    for column in df.columns:
        new_name = (
            column.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        if column != new_name:
            df = df.withColumnRenamed(column, new_name)

    clean_pt_data[name] = df

    print(
        f"{name}: {df.count()} records | "
        f"Removed empty columns: {empty_columns}"
    )

In [0]:
for name, df in clean_pt_data.items():

    table_name = f"workspace.default.silver_{name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Saved: {table_name} | Records: {df.count()}")